# StreetPLM — Eixample Street View Analysis

Uses Meta's **PerceptionLM-1B** (`facebook/Perception-LM-1B`) to analyse a Barcelona
Eixample street-view image across a 3×3 quadrant grid in a single inference pass.

## Requirements
- Python 3.10+
- `torch`, `transformers`, `Pillow`, `numpy`
- HuggingFace access to `facebook/Perception-LM-1B` (or a locally cached copy)

Install missing packages with:
```powershell
pip install torch transformers Pillow numpy
```

If the model is not yet cached, authenticate first:
```powershell
huggingface-cli login
```

## 1 — Environment setup

Add the `streetview_analysis` package to the Python path so we can import `PLMAnalyzer`
regardless of where the notebook is opened from.

In [ ]:
import sys
from pathlib import Path

# In a Jupyter notebook __file__ is not defined.
# Resolve the repo root relative to this notebook's expected location:
#   <repo_root>/scripts/Notebook for Street PLM/StreetPLM_Eixample.ipynb
try:
    # Works when the notebook is run as a script (pytest-nbmake, nbconvert, etc.)
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    # Interactive Jupyter session — use the current working directory.
    # Start Jupyter from the notebook's own directory for this to be correct.
    import os
    NOTEBOOK_DIR = Path(os.getcwd()).resolve()

REPO_ROOT = NOTEBOOK_DIR.parent.parent
STREETVIEW_PKG = REPO_ROOT / "scripts" / "streetview_analysis"

if str(STREETVIEW_PKG) not in sys.path:
    sys.path.insert(0, str(STREETVIEW_PKG))

print(f"Repo root     : {REPO_ROOT}")
print(f"PLM package   : {STREETVIEW_PKG}")
print(f"Package exists: {STREETVIEW_PKG.exists()}")

## 2 — Choose an image

Point `IMAGE_PATH` at any local street-view JPEG.  
A sample image from the Eixample grid is included in the repository at  
`scripts/streetview_analysis/output/images/sv_41.387400_2.168600_h0.jpg`.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# ── Edit this path to point at your own street-view image ──────────────────
IMAGE_PATH = REPO_ROOT / "scripts" / "streetview_analysis" / "output" / "images" / "sv_41.387400_2.168600_h0.jpg"
# ───────────────────────────────────────────────────────────────────────────

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Image not found: {IMAGE_PATH}\n"
        "Run scripts/streetview_analysis/run_analysis.py first, or update IMAGE_PATH above."
    )

img = Image.open(IMAGE_PATH)
print(f"Image: {IMAGE_PATH.name}  ({img.size[0]}×{img.size[1]} px, mode={img.mode})")
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis("off")
plt.title(IMAGE_PATH.name)
plt.tight_layout()
plt.show()

## 3 — Trial cell: load PLMAnalyzer and run analysis

This cell loads the model (once) and performs a single-pass 9-quadrant analysis of the
image chosen above.  
The first run downloads the model weights (~2 GB) if they are not already cached.

In [ ]:
import json
from plm_analyzer import PLMAnalyzer

# Load the model (CPU by default; set device='cuda' if a GPU is available)
analyzer = PLMAnalyzer()

print("\nRunning 9-quadrant urban analysis …")
result = analyzer.analyze_image(IMAGE_PATH)

print(f"\nLatency : {result.get('_latency_ms', '?')} ms")
print(f"Parse OK: {'_parse_error' not in result}")
print("\nRaw model output (first 300 chars):")
print(result.get("_raw", "")[:300])

## 4 — Inspect quadrant results

In [ ]:
QUADRANT_NAMES = [
    "top_left",    "top_center",    "top_right",
    "middle_left", "middle_center", "middle_right",
    "bottom_left", "bottom_center", "bottom_right",
]

for name in QUADRANT_NAMES:
    q = result.get(name, {})
    narrative = q.get("narrative", "(no narrative)")
    typology  = q.get("building_typology", "unknown")
    style     = q.get("architectural_style", "unknown")
    print(f"\n{'─'*60}")
    print(f"  {name.upper()}")
    print(f"{'─'*60}")
    print(f"  Typology : {typology}")
    print(f"  Style    : {style}")
    print(f"  Narrative: {narrative}")

## 5 — Visualise the 3×3 grid with analysis labels

In [ ]:
import numpy as np

img_arr = np.array(img)
h, w = img_arr.shape[:2]
row_h, col_w = h // 3, w // 3

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
grid_pos = [
    ("top_left",    0, 0), ("top_center",    0, 1), ("top_right",    0, 2),
    ("middle_left", 1, 0), ("middle_center",  1, 1), ("middle_right", 1, 2),
    ("bottom_left", 2, 0), ("bottom_center",  2, 1), ("bottom_right", 2, 2),
]

for name, row, col in grid_pos:
    tile = img_arr[row*row_h:(row+1)*row_h, col*col_w:(col+1)*col_w]
    ax = axes[row][col]
    ax.imshow(tile)
    ax.axis("off")
    q = result.get(name, {})
    label = f"{name}\n{q.get('building_typology','?')}"
    ax.set_title(label, fontsize=8, pad=3)

fig.suptitle("PerceptionLM-1B  —  3×3 Quadrant Analysis", fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Save results to JSON

In [ ]:
from datetime import datetime, timezone

output_dir = REPO_ROOT / "scripts" / "streetview_analysis" / "output" / "results"
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
stem = Path(IMAGE_PATH).stem
out_path = output_dir / f"{stem}_{timestamp}.json"

payload = {
    "metadata": {
        "timestamp": timestamp,
        "source_image": Path(IMAGE_PATH).name,
        "source_path": str(IMAGE_PATH),
        "model": "facebook/Perception-LM-1B",
        "device": analyzer.device,
    },
    "quadrant_analysis": result,
}

out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Result saved → {out_path}")